In [1]:
from torchvision.datasets import ImageFolder 
from PIL import Image, ImageOps
import numpy as np
import os
import shutil
from tqdm import tqdm

In [ ]:



def exif_fix(img: Image.Image) -> Image.Image:
    try:
        return ImageOps.exif_transpose(img)
    except Exception:
        return img


def trim_white_edges(img: Image.Image, tol: int = 250, edge_white_frac: float = 0.995) -> Image.Image:
    """Opzionale: rimuove margini bianchi dai bordi esterni."""
    im = img.convert("RGB")
    arr = np.asarray(im)
    h, w, _ = arr.shape
    white = (arr[...,0] >= tol) & (arr[...,1] >= tol) & (arr[...,2] >= tol)

    top, bottom, left, right = 0, h-1, 0, w-1
    while top < bottom and white[top, :].mean() >= edge_white_frac:   top += 1
    while bottom > top and white[bottom, :].mean() >= edge_white_frac: bottom -= 1
    while left < right and white[:, left].mean() >= edge_white_frac:   left += 1
    while right > left and white[:, right].mean() >= edge_white_frac:  right -= 1

    if right <= left or bottom <= top:
        return im
    return im.crop((left, top, right+1, bottom+1))

def warp_to_square_noresize(img: Image.Image, do_trim=False, tol=250) -> Image.Image:
    """
    Warp a quadrato SENZA ridimensionare a 224.
    Risultato: lato = max(width, height) originale.
    """
    im = exif_fix(img).convert("RGB")
    if do_trim:
        im = trim_white_edges(im, tol=tol)
    w, h = im.size
    s = max(w, h)
    return im.resize((s, s), resample=Image.Resampling.BICUBIC)



def letterbox_black_clip(img: Image.Image, trim_edges_first=True, tol=250) -> Image.Image:
    """
    - Rimuove i bordi bianchi esterni.
    - Crea un canvas quadrato nero con lato = max(w, h) dell'immagine.
    - Incolla l'immagine originale centrata, SENZA scalarla.
    """
    im = exif_fix(img).convert("RGB")
    if trim_edges_first:
        im = trim_white_edges(im, tol=tol)

    w, h = im.size
    s = max(w, h)  # lato del canvas quadrato

    canvas = Image.new("RGB", (s, s), (0, 0, 0))
    off_x = (s - w) // 2
    off_y = (s - h) // 2
    canvas.paste(im, (off_x, off_y))
    return canvas





In [3]:
script_dir = os.path.dirname(os.path.abspath(os.path.abspath('')))
DATASETS_BASE_PATH: str = os.path.join(script_dir,"datasets/")

ds_name = "architectural-styles-dataset"
preprocessing=warp_to_square_noresize
ds_name_new = ds_name+ "-spl-warp-lanczos"

desired_splits={"train":0.84, "valid": 0.08, "test": 0.08}


IFds = ImageFolder(os.path.join(DATASETS_BASE_PATH,ds_name)) # letterbox_black_clip #None

num_classes = len(IFds.classes)



img_slpits={}
# 
# extract from the dataset the class of each sample and memorize for each class it's samples' indexes
# class_indexes has dims (num_classes, ?) where ? is the number of samples for each class
#

classes_indexes: list[list[int]] = []
for i in range(num_classes):
    classes_indexes.append([])

i = 0
for t in IFds.targets:
    classes_indexes[t].append(i)
    i+= 1
    
#
# Count the number of samples per class
#
class_samples_count: list[int] = []
for cl_ind in classes_indexes:

    class_samples_count.append(len(cl_ind))
    
max_num = max(class_samples_count)

#
#   Compute the number of samples for each split
#
#valid_num = int(max_num*ds_config.valid_split)
#train_num = max_num - test_num - valid_num

nums = {}
for spl, val in desired_splits.items():
    nums[spl] = int(val * max_num)

nums_no_train = sum([v if k!="train" else 0 for k,v in nums.items()])
nums["train"] = max_num - nums_no_train
#
# check if any class has less samples than the sum of the test + valid splits
#
for count in class_samples_count:
    if count < nums_no_train:
        raise Exception("Dataset Split exception: The number of samples for one of the classes is lower than test + valid")

#
#   Generate the permutations to use for test valid train splits
#
randoms_classes : list[np.ndarray] = []

for num in class_samples_count:
    randoms_classes.append(np.random.permutation(num))

perms = {}
tot = 0
for k,v in nums.items():
    if k != "train":
        perms[k] = np.concatenate([np.array(inds)[r[tot : tot+v]] for r, inds in zip(randoms_classes, classes_indexes)])
        tot+= v
# if self.oversampling :
#     for num, i in zip(class_samples_count, range(num_classes)):
#         to_resample = nums["train"] - (num - nums_no_train )
#         randoms_classes[i]=np.append(randoms_classes[i], np.random.choice(randoms_classes[i][ nums_no_train : ] , to_resample))

perms["train"]= np.concatenate([np.array(inds)[r[nums_no_train : ]] for r, inds in zip(randoms_classes, classes_indexes)])

save_root: str = os.path.join(script_dir,"datasets/",ds_name_new)




for split_name, indices in perms.items():
    for idx in tqdm(indices.tolist()):
        # Get the file path and label
        path, class_id = IFds.samples[idx]
        class_name = IFds.classes[class_id]

        # Destination folder
        dest_dir = os.path.join(save_root, split_name, class_name)
        os.makedirs(dest_dir, exist_ok=True)
        
        # # Copy the file
        # shutil.copy(path, dest_dir)
        file_name = os.path.basename(path)

        dest_path = os.path.join(dest_dir, file_name)
        
        preprocessing(Image.open(path)).save(dest_path)






100%|██████████| 7263/7263 [02:23<00:00, 50.67it/s]
